<a href="https://colab.research.google.com/github/mubashira18/LLM_PROJECTS/blob/main/DocuChat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Research Assistant with RAG

This project implements a research assistant using a Retrieval-Augmented Generation (RAG) system. It combines information retrieval from a specific research paper ("Attention Is All You Need") with web search capabilities to provide comprehensive answers to user queries.

## Features

- **Document Loading & Splitting**: Loads PDF documents and splits them into manageable chunks for efficient processing.
- **Embeddings**: Utilizes pre-trained sentence transformer models to create vector embeddings of document chunks.
- **Vector Store (ChromaDB)**: Stores document embeddings in a Chroma vector database for fast similarity search.
- **Context Retrieval**: Retrieves relevant document chunks based on user queries using similarity search.
- **LLM Integration**: Uses the Gemini API to generate answers based on the retrieved context.
- **Agentic RAG**: Implements an agent that can decide whether to retrieve information from the loaded PDF or perform a web search using TavilySearch, depending on the nature of the query.
- **Gradio UI**: Provides a simple web interface for interacting with the research assistant.

## Setup and Usage

### 1. Install Dependencies

All necessary Python packages are installed in the notebook. Ensure you have activated your Colab environment.

### 2. API Keys

- **GEMINI_API_KEY**: Obtain an API key from Google AI Studio and store it in Colab's secrets manager under the name `GEMINI_API_KEY`.
- **TAVILY_API_KEY**: Obtain an API key from Tavily and store it in Colab's secrets manager under the name `TAVILY_API_KEY`.

### 3. Run the Notebook

Execute all cells in the notebook sequentially. This will:

- Load and process the `self attention.pdf` document.
- Initialize the embedding model and vector store.
- Set up the Gemini LLM and the agent with retrieval and web search tools.
- Launch a Gradio interface.

### 4. Interact with the Assistant

Once the Gradio interface is launched, you can type your questions into the provided text box. The agent will determine the best tool (PDF retrieval or web search) to answer your query and generate a response.

*Loading Document*

In [1]:
!pip install -q langchain-community pypdf

In [2]:
from langchain_community.document_loaders import PyPDFLoader
file_path = '/content/self attention.pdf'
loader = PyPDFLoader(file_path)
doc = loader.load()
print(doc[1].page_content)

1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks
in particular, have been firmly established as state of the art approaches in sequence modeling and
transduction problems such as language modeling and machine translation [ 35, 2, 5]. Numerous
efforts have since continued to push the boundaries of recurrent language models and encoder-decoder
architectures [38, 24, 15].
Recurrent models typically factor computation along the symbol positions of the input and output
sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden
states ht, as a function of the previous hidden state ht−1 and the input for position t. This inherently
sequential nature precludes parallelization within training examples, which becomes critical at longer
sequence lengths, as memory constraints limit batching across examples. Recent work has achieved
significant improvements in computational efficiency through facto

In [3]:
print(doc[1].metadata)

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/self attention.pdf', 'total_pages': 15, 'page': 1, 'page_label': '2'}


*Splitting Document*

In [4]:
!pip install -q langchain-text-splitters

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)
all_splits = text_splitter.split_documents(doc)
print(len(all_splits))

52


In [6]:
print(all_splits[4].page_content)

1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks
in particular, have been firmly established as state of the art approaches in sequence modeling and
transduction problems such as language modeling and machine translation [ 35, 2, 5]. Numerous
efforts have since continued to push the boundaries of recurrent language models and encoder-decoder
architectures [38, 24, 15].
Recurrent models typically factor computation along the symbol positions of the input and output
sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden
states ht, as a function of the previous hidden state ht−1 and the input for position t. This inherently
sequential nature precludes parallelization within training examples, which becomes critical at longer
sequence lengths, as memory constraints limit batching across examples. Recent work has achieved


In [7]:
print(all_splits[5].page_content)

sequential nature precludes parallelization within training examples, which becomes critical at longer
sequence lengths, as memory constraints limit batching across examples. Recent work has achieved
significant improvements in computational efficiency through factorization tricks [21] and conditional
computation [32], while also improving model performance in case of the latter. The fundamental
constraint of sequential computation, however, remains.
Attention mechanisms have become an integral part of compelling sequence modeling and transduc-
tion models in various tasks, allowing modeling of dependencies without regard to their distance in
the input or output sequences [2, 19]. In all but a few cases [27], however, such attention mechanisms
are used in conjunction with a recurrent network.
In this work we propose the Transformer, a model architecture eschewing recurrence and instead
relying entirely on an attention mechanism to draw global dependencies between input and output.


In [8]:
print(f'total splits : {len(all_splits)}')

total splits : 52


*Creating Embeddings*

In [9]:
!pip install -q langchain langchain-huggingface sentence_transformers

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(
    model_name = 'sentence-transformers/all-mpnet-base-v2'
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


*Initializing Database*

In [11]:
!pip install -q langchain-chroma


In [12]:
from langchain_chroma import Chroma
vector_store = Chroma(
    collection_name = 'research_collection',
    embedding_function = embedding_model,
    persist_directory = './chroma_langchain_db'
)

document_ids = vector_store.add_documents(documents=all_splits)
print(document_ids)

['c54cdee7-1112-41e5-b51d-d0ccdd186b40', '9ad2a467-afa1-473f-a1da-425f4d16fcb0', '3ca49959-7197-4c5f-be1c-81a88fa5807e', '11d22326-d40a-43f2-af09-874b6d9f7e10', 'c7cb4ad4-eb66-4246-9401-a03d664689b8', '531628ca-7ec7-4cf8-8d95-9d99719ad09e', '54499c25-aa4a-420a-854d-539edb7c7aeb', '4d14fd17-6da2-4174-99b8-04cebd6befca', '30257e36-d20f-43a7-8bec-8d871bf6c513', '7b01cd04-d238-4111-9668-55e15a2ce912', '7dcb7871-ff24-4385-bf89-fd63b0feaa1d', '402bca02-5b53-417c-9e01-062c1f133aa3', 'bb9a673b-488f-44cd-a86a-fc3ae8590a94', '01758876-8925-4c5a-a184-d13ff4499377', '70dbbbcd-8978-43c4-9e72-3af11067986e', 'd15078f4-939e-4ff5-8836-383dfb899cf9', '23df2669-46e3-4652-9aac-22adca73d36e', 'c4083e78-0480-4a61-9185-638909493145', 'd0b17cbb-ccbf-419b-8539-f413ec12fa16', 'a3c1c3e4-7b44-4de6-8f2e-29c81bb9e94c', 'a26ee2f7-ebf6-407b-980f-a9bece79defa', '8bfe7ef0-900c-4e8d-8910-f3acff3ff1cc', 'cfe4b474-e023-4171-ad7e-1bf995166960', '4a00ff10-611c-41da-913d-52e489ee05de', 'd711cd09-e50a-4cf3-89ee-d31d15961766',

In [13]:
sample = vector_store.get(limit=1,include=['embeddings','documents'])

In [14]:
print(sample)

{'ids': ['c54cdee7-1112-41e5-b51d-d0ccdd186b40'], 'embeddings': array([[ 3.69697786e-03,  1.73237845e-02, -1.36921369e-02,
         3.69928748e-04, -5.26160039e-02, -1.06534199e-03,
         2.71549122e-03, -2.16392726e-02, -6.30408302e-02,
        -3.65784857e-03,  2.98430678e-03, -3.83308530e-02,
         2.39992682e-02,  4.18659449e-02,  5.23187555e-02,
        -1.34343272e-02,  4.05800268e-02,  1.07414946e-02,
        -3.24229859e-02,  2.76800301e-02, -3.64835188e-02,
        -3.69523838e-02,  2.41782106e-02,  4.15229015e-02,
         3.17309573e-02,  7.77823431e-03,  1.27261819e-03,
        -3.48118059e-02,  3.00640594e-02, -3.74134444e-02,
         9.57105309e-03,  3.81201394e-02,  2.73413863e-02,
         6.72826022e-02,  2.13951148e-06, -1.13877198e-02,
         2.13783477e-02, -3.49598713e-02,  1.37139596e-02,
         5.53465681e-04, -3.01515423e-02, -3.72831523e-02,
        -1.29216891e-02, -8.13812483e-03, -2.99346801e-02,
        -4.24709963e-03,  7.07999766e-02,  4.626090

*retrieve and context*

In [15]:
def retrieve_context(query,k=2):
  retrieved_docs = vector_store.similarity_search(query,k=k)
  docs_content = ''
  for doc in retrieved_docs:
    docs_content+= f'Source: {doc.metadata}\n'
    docs_content += f'Content: {doc.page_content}\n\n'
  return docs_content,retrieved_docs

*Generate Answers Using LLM*

In [18]:
!pip install -U langchain-google-genai
from langchain.chat_models import init_chat_model
from google.colab import userdata
api_key = userdata.get('GEMINI_API_KEY')
model = init_chat_model(
    'google_genai:gemini-2.5-flash',
    api_key = api_key
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 1.4 MB/s eta 0:00:00


In [30]:
def docu_chat(user_query):
  context,source_docs = retrieve_context(user_query)
  system_message = f'''you are helpful chatbot.use only the following
  pieces of context to answer the question , don't make up any new information,get as much as possible information from the provided context: {context}'''
  messages = [
      {'role':'system','content':system_message},
      {'role':'user','content':user_query}
  ]
  response = model.invoke(messages)
  return{
      'answer':response.content,
      'source_documents':source_docs,
      'content_used':context
  }


In [21]:
user_query = input()

what is self attention


In [23]:
from pprint import pprint

In [31]:
pprint(docu_chat(user_query).get('answer'))

('An attention function maps a query and a set of key-value pairs to an '
 'output, which is computed as a weighted sum. In terms of computational '
 'complexity, self-attention layers are faster than recurrent layers when '
 'processing sequences.')


*converting this into research assisstant(Agentic RAG)*

In [32]:
system_prompt = """You are a helpful research assistant with access to two tools:

1. retrieve_from_pdf: Use this to find information from the
   "Attention Is All You Need" research paper

2. TavilySearch: Use this to find current information
   not in the paper (recent events, updates, etc.)

Strategy:
- For questions about the paper content → use retrieve_from_pdf
- For questions about recent events or topics not in the paper → use TavilySearch
- DON'T make up things
"""

In [33]:
from langchain.agents import create_agent
agent = create_agent(
    model = model,
    tools = [],
    system_prompt = system_prompt
)

In [ ]:
# @tool
# def function_name(parameter):
#   return

In [35]:
from langchain.tools import tool
@tool
def retrieve_from_pdf(query):
  """Retrieve information from the Attention Is All You Need research paper."""
  retrieved_docs = vector_store.similarity_search(query,k=2)
  docs_content = ''
  for doc in retrieved_docs:
    docs_content+= f'Source: {doc.metadata}\n'
    docs_content += f'Content: {doc.page_content}\n\n'
  return docs_content

In [36]:
!pip install langchain-tavily

In [37]:
from langchain_tavily import TavilySearch
TAVILY_API_KEY = userdata.get('TAVILY_API_KEY')

In [38]:
web_search_tool = TavilySearch(
    max_resutls = 5,
    search_depth = 'advanced',
    tavily_api_key = TAVILY_API_KEY
)

In [43]:
user_query = """Compare the attention mechanism from the paper with recent improvements like Flash Attention, and tell me which approach would be better for my college project."""
response = agent.invoke({
    'messages':[{'role':'user','content':user_query}]
})
pprint(response['messages'][-1].content)

('To compare the attention mechanism from the "Attention Is All You Need" '
 'paper with recent improvements like Flash Attention, and to recommend an '
 "approach for your college project, let's break down each component:\n"
 '\n'
 '### Attention Mechanism from "Attention Is All You Need"\n'
 '\n'
 'The "Attention Is All You Need" paper introduced the Transformer '
 'architecture, which relies heavily on a mechanism called **Multi-Head '
 'Self-Attention**.\n'
 '\n'
 '1.  **Core Idea:** It allows the model to weigh the importance of different '
 'parts of the input sequence when encoding a particular element. For example, '
 'when translating "The animal didn\'t cross the street because it was too '
 'tired," the "it" refers to "animal." Attention helps the model identify this '
 'relationship.\n'
 '2.  **How it Works (Scaled Dot-Product Attention):**\n'
 '    *   It takes a query (Q), keys (K), and values (V) as input.\n'
 '    *   It computes the dot products of the query with all k

*creating ui for docuchat*

In [46]:
def generate_answer(query):
  response = agent.invoke({
      'messages':[
          {'role':'user','content':query}
      ]
  })
  return response['messages'][-1].content


In [47]:
query = input()

flash attention


In [49]:
pprint(generate_answer(query) )

('Flash Attention is a more recent development in optimizing the attention '
 'mechanism, not something discussed in the original "Attention Is All You '
 'Need" paper.\n'
 '\n'
 'Let me search for information about Flash Attention using TavilySearch.\n'
 'Flash Attention is a novel algorithm designed to speed up and reduce the '
 'memory footprint of the attention mechanism, particularly in large language '
 'models. It achieves this by reordering the computation and leveraging tiling '
 'and other techniques to reduce the number of memory accesses to '
 'high-bandwidth memory (HBM).\n'
 '\n'
 'Key benefits of Flash Attention include:\n'
 '*   **Faster training and inference:** It can make attention computations '
 'significantly faster, especially for long sequences.\n'
 '*   **Reduced memory usage:** It avoids materializing the large intermediate '
 'attention matrix (NxN, where N is sequence length), which is often the '
 'bottleneck for memory. This allows for training models with

In [44]:
!pip install gradio

In [45]:
import gradio as gr

In [51]:
demo = gr.Interface(
    fn = generate_answer,
    inputs = [gr.Textbox()],
    outputs = [gr.TextArea()]
)

In [ ]:
demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9081da11caefb2bd4e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Created dataset file at: .gradio/flagged/dataset1.csv
